--------
Read data from silver Minio 
--------
--------

In [4]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
from pyspark.sql.functions import sum
import os
import psycopg2

# -------------------------
# Load env
# -------------------------
load_dotenv()

minio_endpoint = os.getenv("MINIO_ENDPOINT")
minio_access = os.getenv("MINIO_ACCESS_KEY")
minio_secret = os.getenv("MINIO_SECRET_KEY")

# -------------------------
# Spark session
# -------------------------
spark = SparkSession.builder \
    .appName("Olist Aggregation") \
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4"
    ) \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{minio_endpoint}") \
    .config("spark.hadoop.fs.s3a.access.key", minio_access) \
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# -------------------------
# Read Silver from MinIO
# -------------------------
df = spark.read.parquet("s3a://olist-data/silver/fact_orders")

# -------------------------
# Create Gold dataset
# -------------------------
df_gold = df.groupBy("customer_state") \
    .agg(sum("payment_value").alias("total_revenue"))

df_gold.show()

# -------------------------
# Convert to Pandas
# -------------------------
df_pd = df_gold.toPandas()


+--------------+------------------+
|customer_state|     total_revenue|
+--------------+------------------+
|            SC| 786343.7099999995|
|            RO|           65886.0|
|            PI|         136779.96|
|            AM|           34753.3|
|            RR|          12462.21|
|            GO|513878.99999999994|
|            TO|          72281.17|
|            MT|256804.61999999997|
|            SP| 7597209.660000009|
|            PB|180984.18999999997|
|            ES|405805.33999999985|
|            RS| 1147276.999999999|
|            MS|         164337.28|
|            AL|111284.41999999998|
|            MG|2326151.6400000025|
|            PA|261788.34999999998|
|            BA| 797410.3600000002|
|            SE|          88437.51|
|            PE|376377.26999999996|
|            CE|343847.82999999996|
+--------------+------------------+
only showing top 20 rows



--------
Create table and load  to postgresql
--------
--------

In [5]:
# -------------------------
# Load PostgreSQL config from .env
# -------------------------
pg_host = os.getenv("POSTGRES_HOST")
pg_db = os.getenv("POSTGRES_DB")
pg_user = os.getenv("POSTGRES_USER")
pg_password = os.getenv("POSTGRES_PASSWORD")
pg_port = os.getenv("POSTGRES_PORT")


# -------------------------
# Connect to PostgreSQL
# -------------------------
conn = psycopg2.connect(
    host=pg_host,
    database=pg_db,
    user=pg_user,
    password=pg_password,
    port=pg_port
)

cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS revenue_by_state (
    customer_state TEXT,
    total_revenue FLOAT
)
""")

# Clear old data
cursor.execute("DELETE FROM revenue_by_state")

# Insert new data
for _, row in df_pd.iterrows():
    cursor.execute("""
        INSERT INTO revenue_by_state (customer_state, total_revenue)
        VALUES (%s, %s)
    """, (row["customer_state"], row["total_revenue"]))

conn.commit()
cursor.close()
conn.close()

print("✅ Data loaded into PostgreSQL")

✅ Data loaded into PostgreSQL


--------
Check gold layer from postgresql
--------
--------

In [11]:
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv("../.env.local")

conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST"),
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT")
)

cursor = conn.cursor()

cursor.execute("SELECT * FROM revenue_by_state LIMIT 5;")

rows = cursor.fetchall()

for row in rows:
    print(row)

cursor.close()
conn.close()

UndefinedTable: relation "revenue_by_state" does not exist
LINE 1: SELECT * FROM revenue_by_state LIMIT 5;
                      ^
